In [ ]:
# Notebook for data cleaning for release

import json
import pathlib
import copy
import tqdm

datasets_for_release = {
    "allenai_multipref": "multipref_gpt4_human_merged_annotated_ap.json",
    "chatbot_arena": "lmsys_arena_explorer_data_w_topics_2025-02-23_rand10k_annotated_ap.json",
    "llama4_arena_vs_public_version": "llama4_exp_vs_public_vs_other_v2_annotated_filtered_ap.json",
    "model_comparison": "model_comparison_annotated_ap.json",
    "prism": "prism_rand_incl_metadata_v2_annotated_ap.json",
    "human_annotations": "human/arena_w_human_plus_ai_anns_ap_seed0.json"
}

new_path = pathlib.Path("../data/paper")
old_path = pathlib.Path("/Users/arduin/main/repos/huggingface/feedback-forensics-annotations/archive/gpt4omini_annotation_data")
output_path = pathlib.Path("/Users/arduin/main/repos/huggingface/feedback-forensics-annotations/data/")

for dataset_name, file_name in datasets_for_release.items():
    print(f"Processing {dataset_name}")
    new_file_path = new_path / file_name
    old_file_path = old_path / f"{dataset_name}.json"
    output_file_path = output_path / f"{dataset_name}.json"

    with open(new_file_path, "r") as f:
        new_data = json.load(f)

    if old_file_path.exists():
        with open(old_file_path, "r") as f:
            old_data = json.load(f)
    else:
        old_data = new_data

    allowed_metadata = old_data["metadata"]["available_metadata_keys_per_comparison"]

    if dataset_name == "model_comparison" and "prompt_id" in allowed_metadata:
        allowed_metadata.remove("prompt_id")

    new_data["metadata"]["description"] = old_data["metadata"]["description"] + " Annotated with Gemini-2.5-Flash."
    new_data["metadata"]["dataset_name"] = dataset_name
    new_data["metadata"]["available_metadata_keys_per_comparison"] = allowed_metadata

    # delete responses and
    for comparison in tqdm.tqdm(new_data["comparisons"]):
        response_length_a = len(comparison["response_a"]["text"])
        response_length_b = len(comparison["response_b"]["text"])

        for response in ["response_a", "response_b"]:
            comparison[response]["text"] = None
        comparison["prompt"] = None
        comparison["metadata"] = {k: v for k, v in comparison["metadata"].items() if k in allowed_metadata}
        comparison["metadata"]["response_a_length"] = response_length_a
        comparison["metadata"]["response_b_length"] = response_length_b
        comparison["metadata"]["model_a"] = comparison["response_a"].get("model", None)
        comparison["metadata"]["model_b"] = comparison["response_b"].get("model", None)

        # check that all metadata keys are present
        for key in allowed_metadata:
            assert key in comparison["metadata"], f"Metadata key {key} not found in {dataset_name}"

    with open(output_file_path, "w") as f:
        json.dump(new_data, f, indent=4)











